# Figure 2 — training loss as the bags grow

Two panels, one per $d$, at fixed $\alpha=\ln n/d$: the training loss against
epoch for runs on each dataset, coloured by initial overlap $u_0$.

Training is full batch, so one gradient step is one epoch. The trajectory
cells store both the step count and the flow time $t=\mathrm{lr}\cdot
\mathrm{step}$; this figure plots the step count.

Data comes from `extremal_mil.fig2_experiment`; this notebook only reads and
plots:

```
python -m extremal_mil.fig2_experiment
```

In [ ]:

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import gridspec
from matplotlib.patches import FancyArrowPatch, Rectangle

import sys
sys.path.insert(0, str(Path.cwd().parent / "src"))
from extremal_mil import data, fig2_experiment, plotting

# shared paper style; the figure is drawn at its printed width (one column)
plt.rcParams.update(plotting.RCPARAMS)

get_ipython().run_line_magic('config', "InlineBackend.figure_format = 'svg'")

FIG_DIR = Path.cwd().parent / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Colors used consistently in both panels.
FEATURE_COLOR = "#c8860d"   # gold: the feature picked out by beta = e_1
INSTANCE_COLOR = "#2f6fab"  # blue: the instance picked by arg max (MIL only)

CMAP = plt.cm.Greys
NORM = mcolors.Normalize(vmin=-2.3, vmax=2.3)

# Shared row layout so the two panels line up: row/bag i occupies the same
# vertical slot in both panels regardless of what's drawn inside it.
N_ROWS = 4
X0, Y_TOP, GAP, SLOT_H = 0.08, 0.85, 0.05, 0.14

In [ ]:
# Loss trajectories at fixed ell, with d -- and n = e^{alpha d} with it --
# growing. Both panels run one schedule, so they cover the same epochs.
ELL = 100
# d=22 rather than 20: n is rounded to an integer, so d=20 gives n=3 and an
# effective alpha of 0.0549, 9.9% above nominal, where d=22 gives 0.0499.
D_SHOW = [22, 150]
# lr=0.1: at lr=0.3 the (rho, lambda) path is visibly jagged, lr=0.2 still is.
SCHEDULE = (0.1, 150000)


def load_cell(d):
    # One schedule, no fallback. Falling back to another lr or horizon when a
    # cell is missing silently changes the data the figure shows, which is how
    # the panels came to be drawn from different ell.
    lr, n_steps = SCHEDULE
    return fig2_experiment.load_trajectory_sweep("u", ELL, d, lr, n_steps)


traj = {d: load_cell(d) for d in D_SHOW}
for d, t in traj.items():
    n = int(t["n"])
    print(f"d={d:<4d} n={n:<6d} alpha_eff=ln(n)/d={np.log(n) / d:.4f}  "
          f"lr={float(t['lr']):<5g} steps={int(t['n_steps']):<7d} "
          f"t up to {float(t['flow_time']):g}")

x0 = traj[D_SHOW[0]]["x0"]
colors, mappable = plotting.run_colors(x0)
t_max = max(float(t["flow_time"]) for t in traj.values())
loss_lim = (min(t["loss"].min() for t in traj.values()) / 2,
            max(t["loss"].max() for t in traj.values()) * 2)

# One x-range for both panels, so a run that stops earlier shows as a curve
# ending early rather than as a panel with its own scale.
epoch_lim = (min(t["step"][t["step"] > 0].min() for t in traj.values()),
             max(t["step"].max() for t in traj.values()))

fig, axes = plt.subplots(1, len(D_SHOW), figsize=(plotting.COL_W, 1.9), sharey=True,
                         sharex=True, layout="constrained")

for j, (ax, d) in enumerate(zip(np.atleast_1d(axes), D_SHOW)):
    t = traj[d]
    # Full-batch training, so one gradient step is one epoch; the cells store
    # the step count alongside the flow time t = lr * step.
    epoch = t["step"]
    keep = epoch > 0
    for i, c in enumerate(colors):
        ax.plot(epoch[keep], t["loss"][i][keep], color=c, lw=0.9, alpha=0.9)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(*loss_lim)
    ax.set_xlim(*epoch_lim)
    ax.set_xlabel("epoch")
    ax.set_title(f"({'ab'[j]})", pad=3)
    if j:
        ax.tick_params(labelleft=False)
    else:
        ax.set_ylabel("loss")

cbar = fig.colorbar(mappable, ax=list(np.atleast_1d(axes)), fraction=0.04, pad=0.02)
cbar.set_label("$u_0$")
cbar.outline.set_visible(False)

fig.savefig(FIG_DIR / "fig2.pdf", bbox_inches="tight")
plt.show()
